# Federated Medical Image Classification — Multi-Dataset Experiments

**ResNet18 · CE + Grid-wise Perona-Malik PIDL · Flower SecAgg+ · No DP noise**

---

**Workflow**
1. Clone the project from GitHub
2. Download public Kaggle datasets (no API key needed)
3. Run `flwr run .` for each experiment
4. Download `results/` at the end — push to git from your local PC

| Section | Contents |
|---|---|
| 1 | Clone repo + install dependencies |
| 2 | Download datasets (public, no auth) |
| 3 | Preview dataset structure |
| 4 | Experiment configuration |
| 5 | Run experiment loop |
| 6 | Verify outputs + master summary |
| 7 | Quick plots |
| 8 | Download results |

> **Resume**: runs whose `fl_summary.json` already exists are skipped.  
> **GPU cache** is cleared between runs automatically.

---
## § 1 — Clone Repo + Install Dependencies

In [ ]:
# ── Edit your GitHub repo URL here ───────────────────────────────────
GITHUB_REPO = 'https://github.com/PulockDas/medical_fl_pidl.git'
# ──────────────────────────────────────────────────────────────────────

import os, sys
from pathlib import Path

PROJECT_ROOT = Path('/content/medical_fl_pidl')

if not PROJECT_ROOT.exists():
    os.system(f'git clone {GITHUB_REPO} {PROJECT_ROOT}')
else:
    print('Repo already cloned. Pulling latest...')
    os.system(f'git -C {PROJECT_ROOT} pull')

if not PROJECT_ROOT.exists():
    raise RuntimeError(
        f'Clone failed. Check that GITHUB_REPO is set correctly: {GITHUB_REPO!r}'
    )

sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)   # flwr run . needs pyproject.toml in cwd
print(f'Working dir: {Path.cwd()}')


In [ ]:
# Step 1: upgrade pip + setuptools (Colab ships an old setuptools that
#         breaks editable installs with the Flower build backend)
!pip install -q --upgrade pip setuptools wheel

# Step 2: install all project dependencies from requirements.txt
#         (avoids the pip install -e . build-backend issue entirely)
!pip install -q -r requirements.txt

# Quick sanity check
!python -c "import flwr, torch, kagglehub; print(f'flwr={flwr.__version__}  torch={torch.__version__}  kagglehub OK')"
print('Dependencies installed.')


---
## § 2 — Download Datasets

These are **public** Kaggle datasets. `kagglehub` downloads them without any API key or `kaggle.json`.  
Subsequent runs use the local cache — no re-download.

| Dataset | Slug | Structure note |
|---|---|---|
| Brain Tumor MRI | `masoudnickparvar/brain-tumor-mri-dataset` | `Training/class_folders/` — Strategy 1 |
| Lung & Colon Cancer | `andrewmvd/lung-and-colon-cancer-histopathological-images` | Two sub-sets; `COLON_OR_LUNG` chooses which |
| COVID-19 Radiography | `tawsifurrahman/covid19-radiography-database` | `class/images/` nesting — Strategy 5 auto-flattens |

> To swap datasets: edit the slug strings below **and** update `configs/dataset_configs.py`.

In [ ]:
import kagglehub
from pathlib import Path as _P
from data.kaggle_loader import find_image_root, preview_dataset_structure

# ── Dataset slugs — edit here to switch datasets ───────────────────────
# Slugs are confirmed public Kaggle datasets; no API key needed.
DATASET_SLUGS = {
    'brain_tumor_mri':           'masoudnickparvar/brain-tumor-mri-dataset',
    'colon_cancer_or_pathology': 'andrewmvd/lung-and-colon-cancer-histopathological-images',
    'covid':                     'tawsifurrahman/covid19-radiography-database',
}

# num_classes = 0  →  auto-detected from the folder count at runtime
DATASET_NUM_CLASSES = {
    'brain_tumor_mri':           0,   # auto (4: glioma, meningioma, notumor, pituitary)
    'colon_cancer_or_pathology': 0,   # auto (2: colon_aca, colon_n)
    'covid':                     0,   # auto (4: COVID, Lung_Opacity, Normal, Viral Pneumonia)
}

SEP = '-' * 62
DATA_ROOTS = {}

# ── Brain Tumor MRI ────────────────────────────────────────────────────
# Structure: root/Training/{glioma,meningioma,notumor,pituitary}  (4 classes)
#            find_image_root Strategy 1 detects Training/ directly.
print(SEP)
print('  brain_tumor_mri')
bt_raw = kagglehub.dataset_download('masoudnickparvar/brain-tumor-mri-dataset')
DATA_ROOTS['brain_tumor_mri'] = find_image_root(bt_raw)
print(f'  ImageFolder root: {DATA_ROOTS["brain_tumor_mri"]}')

# ── COVID-19 Radiography Database ─────────────────────────────────────
# Structure: root/COVID-19_Radiography_Dataset/{COVID,Lung_Opacity,Normal,
#            Viral Pneumonia}/images/*.jpg   (images are in a sub-subdir)
#            find_image_root Strategy 5 detects the class/images/ nesting and
#            creates a flat /tmp/ directory with symlinks automatically.
print(SEP)
print('  covid')
covid_raw = kagglehub.dataset_download('tawsifurrahman/covid19-radiography-database')
DATA_ROOTS['covid'] = find_image_root(covid_raw)
print(f'  ImageFolder root: {DATA_ROOTS["covid"]}')

# ── Lung and Colon Cancer Histopathological Images ─────────────────────
# Structure: root/lung_colon_image_set/
#              colon_image_sets/{colon_aca, colon_n}  ← 2 colon classes
#              lung_image_sets/{lung_aca, lung_n, lung_scc}  ← 3 lung classes
# We must choose one subset.  Change COLON_OR_LUNG to 'lung_image_sets' to
# run on lung cancer classes instead.
COLON_OR_LUNG = 'colon_image_sets'   # 'colon_image_sets' | 'lung_image_sets'
print(SEP)
print(f'  colon_cancer_or_pathology  (using {COLON_OR_LUNG})')
lc_raw  = kagglehub.dataset_download('andrewmvd/lung-and-colon-cancer-histopathological-images')
lc_root = _P(lc_raw) / 'lung_colon_image_set' / COLON_OR_LUNG
DATA_ROOTS['colon_cancer_or_pathology'] = str(lc_root)
print(f'  ImageFolder root: {lc_root}')

print(f'\n{SEP}')
print('All datasets ready.')


---
## § 3 — Preview Dataset Structure

In [ ]:
for ds_name, root in DATA_ROOTS.items():
    print('=' * 55)
    print(f'  {ds_name}')
    print('=' * 55)
    preview_dataset_structure(root, max_depth=3)


---
## § 4 — Experiment Configuration

Edit these values to change FL or PIDL settings.  
All are forwarded as `--run-config` overrides to `flwr run .`.

In [ ]:
# ── Experiment grid ────────────────────────────────────────────────────
DATASETS_TO_RUN = list(DATASET_SLUGS.keys())
CLIENT_COUNTS   = [3, 4, 5]

# ── FL training ────────────────────────────────────────────────────────
NUM_SERVER_ROUNDS = 5
LOCAL_EPOCHS      = 2
BATCH_SIZE        = 32
LEARNING_RATE     = 0.001
IMAGE_SIZE        = 224

# ── PIDL regularization ────────────────────────────────────────────────
FEATURE_LAYER    = 'layer2'         # layer1 / layer2 / layer3 / layer4
REGULARIZER_TYPE = 'perona_malik'   # perona_malik | isotropic | none
LAMBDA_PM        = 0.1
USE_GRID_LOSS    = True
GRID_SIZE        = 4
K                = 1.0

# ── SecAgg+ ────────────────────────────────────────────────────────────
SECAGG_MAX_WEIGHT = 1048575    # 2^20 - 1

RANDOM_SEED  = 42
RESULTS_ROOT = PROJECT_ROOT / 'results'
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

print(f'Grid: {len(DATASETS_TO_RUN)} datasets x {len(CLIENT_COUNTS)} client counts = {len(DATASETS_TO_RUN)*len(CLIENT_COUNTS)} runs')
print(f'Results: {RESULTS_ROOT}')


---
## § 5 — Experiment Loop

Calls `flwr run .` as a subprocess for each (dataset, num_clients) pair.  
The server writes all output files to `log_dir` automatically.  
**Already-completed runs are skipped** (resume support).

In [ ]:
import gc, json, subprocess, time, torch


def run_experiment(ds_name: str, num_clients: int, resume: bool = True) -> dict:
    log_dir    = RESULTS_ROOT / ds_name / f'{num_clients}_clients'
    status     = dict(ds_name=ds_name, num_clients=num_clients,
                      log_dir=str(log_dir), skipped=False, success=False)

    # Resume: skip if fl_summary.json already exists
    if resume and (log_dir / 'fl_summary.json').exists():
        print(f'  [SKIP] {ds_name} / {num_clients}c — already done.')
        status.update(skipped=True, success=True)
        return status

    log_dir.mkdir(parents=True, exist_ok=True)
    recon = max(1, num_clients - 1)

    run_config = (
        f'dataset_name={ds_name} '
        f'data_root={DATA_ROOTS[ds_name]} '
        f'num_classes={DATASET_NUM_CLASSES.get(ds_name, 0)} '
        f'num_clients={num_clients} '
        f'min_fit_clients={num_clients} '
        f'num_server_rounds={NUM_SERVER_ROUNDS} '
        f'local_epochs={LOCAL_EPOCHS} '
        f'batch_size={BATCH_SIZE} '
        f'learning_rate={LEARNING_RATE} '
        f'image_size={IMAGE_SIZE} '
        f'feature_layer={FEATURE_LAYER} '
        f'regularizer_type={REGULARIZER_TYPE} '
        f'lambda_pm={LAMBDA_PM} '
        f'use_grid_loss={str(USE_GRID_LOSS).lower()} '
        f'grid_size={GRID_SIZE} '
        f'k={K} '
        f'random_seed={RANDOM_SEED} '
        f'log_dir={log_dir} '
        f'secagg_num_shares={num_clients} '
        f'secagg_reconstruction_threshold={recon} '
        f'secagg_max_weight={SECAGG_MAX_WEIGHT}'
    )

    cmd = ['flwr', 'run', '.', '--federation',
           f'local-simulation-{num_clients}', '--run-config', run_config]

    env = os.environ.copy()
    env['PYTHONPATH'] = str(PROJECT_ROOT)

    print(f'  [RUN ] {ds_name} / {num_clients}c  ->  {log_dir.name}/')
    t0   = time.time()
    proc = subprocess.Popen(
        cmd, cwd=str(PROJECT_ROOT),
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=env,
    )
    lines = []
    kw = ('Round', '[Server]', 'Error', 'ERROR', 'Traceback', 'FAIL', 'Best', 'Final')
    for line in proc.stdout:
        lines.append(line.rstrip())
        if any(k in line for k in kw):
            print(f'    {line.rstrip()}')
    proc.wait()
    elapsed = time.time() - t0

    if proc.returncode != 0:
        print(f'  [FAIL] exit {proc.returncode} after {elapsed:.0f}s')
        for l in lines[-20:]: print(f'    {l}')
    else:
        print(f'  [OK  ] {elapsed:.0f}s')
        status['success'] = True

    if torch.cuda.is_available(): torch.cuda.empty_cache()
    gc.collect()
    return status


print('run_experiment() defined.')


In [ ]:
# ── Main loop ──────────────────────────────────────────────────────────
run_log = []
total, done = len(DATASETS_TO_RUN) * len(CLIENT_COUNTS), 0
t_total = time.time()

for ds in DATASETS_TO_RUN:
    if ds not in DATA_ROOTS:
        print(f'[WARN] No data root for {ds} — skipping.')
        continue
    for nc in CLIENT_COUNTS:
        done += 1
        print(f'\n--- Run {done}/{total}  |  {ds}  |  {nc} clients ---')
        run_log.append(run_experiment(ds, nc, resume=True))

elapsed = time.time() - t_total
ok  = sum(1 for r in run_log if r['success'] and not r['skipped'])
skp = sum(1 for r in run_log if r['skipped'])
err = sum(1 for r in run_log if not r['success'])
print(f'\nDone in {elapsed/60:.1f} min  |  OK={ok}  skipped={skp}  failed={err}')


---
## § 6 — Verify Outputs + Master Summary

In [ ]:
import pandas as pd
from pathlib import Path

EXPECTED = [
    'fl_rounds.csv', 'fl_summary.json',
    'per_class_metrics.csv', 'config.json',
    'dataset_summary.json', 'round_metrics.jsonl',
]

print(f'  {"Run":<38}  {"  ".join(f[:10] for f in EXPECTED)}')
print('  ' + '-' * 90)
for r in run_log:
    d = Path(r['log_dir'])
    c = ['OK' if (d / f).exists() else '--' for f in EXPECTED]
    print(f'  {r["ds_name"][:18]}/{r["num_clients"]}c   {"  ".join(c)}')


In [ ]:
# Build master_summary.csv from fl_summary.json files
rows = []
for ds in DATASETS_TO_RUN:
    for nc in CLIENT_COUNTS:
        p = RESULTS_ROOT / ds / f'{nc}_clients' / 'fl_summary.json'
        if not p.exists(): continue
        s = json.load(open(p))
        s.update(dataset_name=ds, num_clients=nc)
        rows.append(s)

if rows:
    master = pd.DataFrame(rows)
    master.to_csv(RESULTS_ROOT / 'master_summary.csv', index=False)
    cols = ['dataset_name', 'num_clients', 'best_accuracy', 'best_macro_f1', 'final_ece']
    avail = [c for c in cols if c in master.columns]
    print(master[avail].to_string(index=False, float_format='{:.4f}'.format))
else:
    print('No completed runs yet.')


---
## § 7 — Quick Plots

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

COLORS  = {3: '#1f77b4', 4: '#ff7f0e', 5: '#2ca02c'}
MARKERS = {3: 'o', 4: 's', 5: '^'}

# ── Per-round curves ────────────────────────────────────────────────────
for ds in DATASETS_TO_RUN:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    fig.suptitle(ds.replace('_', ' ').title(), fontsize=12, fontweight='bold')
    for nc in CLIENT_COUNTS:
        csv = RESULTS_ROOT / ds / f'{nc}_clients' / 'fl_rounds.csv'
        if not csv.exists(): continue
        df = pd.read_csv(csv)
        for ax, col, title in zip(
            axes,
            ['global_test_acc', 'f1_macro', 'global_test_loss'],
            ['Accuracy', 'Macro F1', 'Loss'],
        ):
            if col not in df.columns: continue
            ax.plot(df['round'], df[col], color=COLORS[nc],
                    marker=MARKERS[nc], label=f'{nc}c', markersize=5)
            ax.set_title(title); ax.set_xlabel('Round')
            ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    axes[0].legend()
    plt.tight_layout()
    plt.savefig(RESULTS_ROOT / ds / 'curves.png', dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
# ── Final accuracy + F1 bar charts ─────────────────────────────────────
if 'master' in dir() and not master.empty:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, col, title in zip(
        axes,
        ['best_accuracy', 'best_macro_f1'],
        ['Best Accuracy', 'Best Macro F1'],
    ):
        if col not in master.columns: continue
        pivot = (
            master[['dataset_name', 'num_clients', col]]
            .assign(num_clients=lambda d: d['num_clients'].astype(str) + 'c')
            .pivot(index='dataset_name', columns='num_clients', values=col)
        )
        pivot.plot(kind='bar', ax=ax, colormap='tab10',
                   edgecolor='black', linewidth=0.5, width=0.7)
        ax.set_title(title, fontweight='bold')
        ax.set_ylim(0, 1.08); ax.set_xlabel('')
        ax.set_xticklabels([x.replace('_', '\n') for x in pivot.index], rotation=0)
        for ctr in ax.containers:
            ax.bar_label(ctr, fmt='%.3f', fontsize=7, padding=2)
    plt.tight_layout()
    plt.savefig(RESULTS_ROOT / 'final_bars.png', dpi=150, bbox_inches='tight')
    plt.show()


---
## § 8 — Download Results

Zip the `results/` folder and download it to your local machine.  
Then `git add results/ && git commit && git push` from your PC.

In [ ]:
# Zip results/ and trigger browser download
!zip -r /content/fl_results.zip {RESULTS_ROOT}

from google.colab import files
files.download('/content/fl_results.zip')
print('Download started. Save the zip, unzip into your local repo, then push to git.')


---
## End

**Local git workflow after downloading:**
```bash
cd medical_fl_pidl/
unzip ~/Downloads/fl_results.zip -d .
git add results/
git commit -m 'add experiment results'
git push
```

Then open `02_result_analysis_and_plots.ipynb` in Colab to generate publication-quality figures from the saved results.